# Trabajo Practico Big Data

In [0]:
%restart_python

In [0]:
#importamos librerias

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import col, countDistinct
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# MLflow
import mlflow
from mlflow.models.signature import ModelSignature, infer_signature
from mlflow.types.schema import Schema, TensorSpec, ColSpec

# General
import numpy as np
import os
import shutil

# Optuna
%pip install optuna
import optuna


#configuro session timeout de databricks
spark.conf.set("spark.databricks.execution.timeout", 3600)

### Importamos el dataset

In [0]:
#cargamos el dataset de fraudes bancarios

df = spark.table('forecast.default.data')
df.describe().show()
df.printSchema()
df.select(countDistinct("type")).show()
df.show(10)


### Procesamiento del dataset

In [0]:
# 1. Preparación de datos

df = df.withColumn("isFraud", col("isFraud").cast("double"))

# Feature engineering:

# consistencia de saldos:
df = df.withColumn("errorBalanceOrig", 
    F.col("newbalanceOrig") - F.col("oldbalanceOrg") + F.col("amount"))

df = df.withColumn("errorBalanceDest",
    F.col("newbalanceDest") - F.col("oldbalanceDest") - F.col("amount"))

# cuentas con saldo 0
df = df.withColumn("origZeroBalance",
    ((F.col("oldbalanceOrg") == 0) & (F.col("newbalanceOrig") == 0)).cast("double"))

df = df.withColumn("destZeroBalance", 
    ((F.col("oldbalanceDest") == 0) & (F.col("newbalanceDest") == 0)).cast("double"))

# Indexamos las categorías string
indexer = StringIndexer(
    inputCol="type",
    outputCol="type_idx",
    handleInvalid="keep"
)

numeric_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud',
    # nuevas features
    'errorBalanceOrig', 'errorBalanceDest',
    'origZeroBalance', 'destZeroBalance'
]

feature_cols = numeric_features + ["type_idx"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = indexer.fit(df).transform(df)

df_model = assembler.transform(df).select("features", "isFraud")
df_model = df_model.withColumnRenamed("isFraud", "label")

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42077651)


# 2. Evaluadores

# curva Presisión-Recall
auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

#metric F2 a mano porque no está en pyspark
def compute_f2_fraud(preds, fraud_label=1.0):
    """
    Calcula F2 solo para la clase fraude (label=1),
    ignorando la clase mayoritaria que infla las métricas weighted.
    """
    total_fraud   = preds.filter(F.col("label") == fraud_label).count()
    predicted_pos = preds.filter(F.col("prediction") == fraud_label).count()
    true_pos      = preds.filter(
        (F.col("label") == fraud_label) & (F.col("prediction") == fraud_label)
    ).count()

    precision = true_pos / predicted_pos if predicted_pos > 0 else 0.0
    recall    = true_pos / total_fraud   if total_fraud   > 0 else 0.0

    denom = (4 * precision + recall)
    f2    = 5 * precision * recall / denom if denom > 0 else 0.0

    return f2, precision, recall  # devolvemos los 3 para loggear en MLflow

### Optimizacion de Hiperparametros con Optuna y Registro con MLflow

In [0]:

#seteo directorio para archivos temportales de mlflow
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/forecast/default/prueba"

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/martoschelp@gmail.com/RF_Fraud_Optuna_Spark")


#tomo muestra para el signature de mlflow
n_features = train_df.select("features").first()[0].size

signature = ModelSignature(
    inputs=Schema([TensorSpec(np.dtype("double"), (-1, n_features), "features")]),
    outputs=Schema([
        ColSpec("double", "prediction")
    ])
)

# 4. Función objetivo Optuna (multiobjetivo)

def objective(trial):
    params = {
        "numTrees": trial.suggest_int("numTrees", 50, 200, step=50),
        "maxDepth": trial.suggest_int("maxDepth", 3, 15),
        "maxBins": trial.suggest_int("maxBins", 32, 64, step=8),
        "minInstancesPerNode": trial.suggest_int("minInstancesPerNode", 5, 50),
        "featureSubsetStrategy": trial.suggest_categorical(
            "featureSubsetStrategy", ["sqrt", "log2", "onethird"])
    }

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        seed=42077651,
        **params
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    auc            = auc_eval.evaluate(preds)
    f2, prec, rec  = compute_f2_fraud(preds, fraud_label=1.0)  # 👈 clase fraude

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("pr_auc",   auc)
        mlflow.log_metric("f2_fraud",  f2)
        mlflow.log_metric("precision_fraud", prec)
        mlflow.log_metric("recall_fraud",    rec)
        mlflow.spark.log_model(model, artifact_path="RandomForest",signature=signature)

    return auc, f2


# 5. Optimización multiobjetivo

#semilla para HT con optuna
sampler = optuna.samplers.TPESampler(seed=42077651)

study = optuna.create_study(
    directions=["maximize", "maximize"],
    sampler=sampler  # ← acá va la semilla
)
#optimizamos 20 trials o 100 minutos, lo que pase primero
study.optimize(
    objective,
    n_trials=20,
    timeout=6000  # 100 minutos en segundos, dejás margen para correr el resto del codigo
)


# 6. Ver mejores trials (Pareto front)


# Con multiobjetivo no hay un único "mejor" trial,
# sino una frontera de Pareto
pareto_trials = study.best_trials  # 👈 todos los trials no dominados

print(f"Trials en la frontera de Pareto: {len(pareto_trials)}\n")

for trial in pareto_trials:
    print(f"Trial #{trial.number}")
    print("Params:", trial.params)
    print("AUC:", trial.values[0])
    print("F2: ", trial.values[1])
    print("-" * 50)

# Para elegir UN solo trial,
# rankear por promedio o por la métrica prioritaria
best_by_mean = max(pareto_trials, key=lambda t: (t.values[0] + t.values[1]) / 2)
best_by_f2   = max(pareto_trials, key=lambda t: t.values[1])
best_by_auc  = max(pareto_trials, key=lambda t: t.values[0])

print("\n>> Mejor por promedio AUC+F2:")
print(f"   Trial #{best_by_mean.number} | AUC={best_by_mean.values[0]:.4f} | F2={best_by_mean.values[1]:.4f}")

print("\n>> Mejor priorizando F2:")
print(f"   Trial #{best_by_f2.number}   | AUC={best_by_f2.values[0]:.4f} | F2={best_by_f2.values[1]:.4f}")

print("\n>> Mejor priorizando AUC:")
print(f"   Trial #{best_by_auc.number}  | AUC={best_by_auc.values[0]:.4f} | F2={best_by_auc.values[1]:.4f}")

In [0]:
import optuna.visualization

#graficos para optuna con multiobjetivo

# Targets con nombres semánticos de tu estudio
target_prauc  = lambda t: t.values[0]  # pr_auc
target_f2     = lambda t: t.values[1]  # f2_fraud

# --- Optimization History ---
optuna.visualization.plot_optimization_history(
    study,
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_optimization_history(
    study,
    target=target_f2,
    target_name="F2 Fraud"
)

# --- Parameter Importances ---
optuna.visualization.plot_param_importances(
    study,
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_param_importances(
    study,
    target=target_f2,
    target_name="F2 Fraud"
)

# --- Parallel Coordinate ---
# Útil para ver cómo se combinan los params en cada trial
optuna.visualization.plot_parallel_coordinate(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],  # 👈 tus 4 params
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_parallel_coordinate(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_f2,
    target_name="F2 Fraud"
)

# --- Slice Plot ---
optuna.visualization.plot_slice(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],  # 👈 tus 4 params
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_slice(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_f2,
    target_name="F2 Fraud"
)

# --- Pareto Front (exclusivo de multi-objetivo) ---
optuna.visualization.plot_pareto_front(
    study,
    target_names=["PR-AUC", "F2 Fraud"]  # 👈 el más importante para tu caso
)

In [0]:
best_trial = best_by_mean

with mlflow.start_run(run_name="RandomForest_Final") as run:
    best_params = best_trial.params

    rf_final = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        seed=42077651,
        **best_params
    )

    model_final = rf_final.fit(train_df)
    preds_final = model_final.transform(test_df)

    auc_final = auc_eval.evaluate(preds_final)
    f2_final, prec_final, rec_final = compute_f2_fraud(preds_final, fraud_label=1.0)

    mlflow.log_params(best_params)
    mlflow.log_metric("pr_auc",          auc_final)
    mlflow.log_metric("f2_fraud",        f2_final)
    mlflow.log_metric("precision_fraud", prec_final)
    mlflow.log_metric("recall_fraud",    rec_final)
    mlflow.log_param("selection_criteria",  "best_mean_pareto")
    mlflow.log_param("optuna_trial_number", best_trial.number)

    mlflow.spark.log_model(
        model_final,
        artifact_path="RandomForest",
        signature=signature
    )

    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/RandomForest",  # ← run actual, no best_run_id
        name="workspace.default.rf_fraud_detection"
    )

    print(f"Modelo final registrado | AUC={auc_final:.4f} | F2={f2_final:.4f}")

In [0]:
#levantar modelo de MLflow

#seteo directorio para archivos temportales de mlflow
#os.environ["MLFLOW_DFS_TMP"] = "/Volumes/forecast/default/prueba"

#mlflow.set_registry_uri("databricks-uc")

# Primero buscás los runs ordenados por la métrica que te interese
#runs = mlflow.search_runs(
#    experiment_names=["/Users/martoschelp@gmail.com/RF_Fraud_Optuna_Spark"]
#)

# Calculás el promedio y ordenás en pandas
#runs["mean_score"] = (runs["metrics.pr_auc"] + runs["metrics.f2_fraud"]) / 2
#runs = runs.sort_values("mean_score", ascending=False)

#best_run = runs.iloc[0]
#best_run_id = best_run["run_id"]

# Levantás el modelo
#model = mlflow.spark.load_model(f"runs:/{best_run_id}/RandomForest")

#best_params = {
#    "numTrees":              int(best_run["params.numTrees"]),
#    "maxDepth":              int(best_run["params.maxDepth"]),
#    "maxBins":               int(best_run["params.maxBins"]),
#    "minInstancesPerNode":   int(best_run["params.minInstancesPerNode"]),
#    "featureSubsetStrategy": best_run["params.featureSubsetStrategy"]
#}

#print(f"Run: {best_run['tags.mlflow.runName']}")
#print(f"AUC: {best_run['metrics.pr_auc']:.4f} | F2: {best_run['metrics.f2_fraud']:.4f}")
#print("Params:", best_params)

In [0]:
# Registrar el el modelo para servir (mejor trial por promedio)

#best_run_name = f"optuna_trial_{best_by_mean.number}"

# Buscar el run_id correspondiente
#runs = mlflow.search_runs(
#    experiment_names=["/Users/martoschelp@gmail.com/RF_Fraud_Optuna_Spark"],
#    filter_string=f"tags.mlflow.runName = '{best_run_name}'"
#)

#best_run_id = runs.iloc[0]["run_id"]

#mlflow.register_model(
#    model_uri=f"runs:/{best_run_id}/RandomForest",
#    name="workspace.default.randomforest_fraud_detection"  # catalog.schema.model
#)